In [ ]:
###use with 'postprocessing_skeletons' branch of ac_segmentation rep
import navis
import numpy as np
from ac_segmentation.reconnect_stack_navis import reconnect, read_navis_neurons_tar, write_navis_skels_tar, translate_nodes, filter_skeletons
from joblib import dump,load, Parallel, delayed, parallel_config
import os
import pandas as pd

In [ ]:
def reconnect_strips(strip1, strip2, overlap, cl=None, sc=None, edge_prop=.8, min_nodes=20, query_dis=20, min_collin=0.6, prob_thresh=0.5, dis_end=0):
    if isinstance(strip1, navis.core.neuronlist.NeuronList):
        s1,s2 = strip1,strip2
        strip1,strip2 = 'strip1','strip2'
        pass
    else:
      if strip1.endswith('.gz'):
        s1 = read_navis_neurons_tar(strip1)
        s2 = read_navis_neurons_tar(strip2)
      elif strip1.endswith('.swc'):
        s1 = navis.read_swc(strip1)
        s2 = navis.read_swc(strip2)
          
    #combine neuronlist pairs and set strip names
    s1.set_neuron_attributes([strip1]*int(len(s1)), 'strip')
    s2.set_neuron_attributes([strip2]*int(len(s2)), 'strip')
    skels = navis.NeuronList([s1,s2])

    #find bounding box
    edge_dis = [edge_prop if i != 0  else i for i in overlap]
    x,y,z = skels.bbox
    x_dis,y_dis,z_dis = ((np.diff(x)*edge_dis[0])/2)[0], ((np.diff(y)*edge_dis[1])/2)[0], ((np.diff(z)*edge_dis[2])/2)[0]
    bound_box = np.concatenate((x + np.array([x_dis,-x_dis]), y + np.array([y_dis,-y_dis]), z + np.array([z_dis,-z_dis]))).tolist()

    #run reconnection
    non_merged, merged = reconnect(skels=skels, cl=cl, sc=sc, min_nodes=min_nodes, query_dis=query_dis, min_collin=min_collin, 
                                   prob_thresh=prob_thresh, resample=None, smooth=None, split=False, bound_box=bound_box, dis_end=dis_end)
    merged.set_neuron_attributes(['merged']*int(len(merged)), 'strip')

    if not isinstance(s1, navis.core.neuronlist.NeuronList):
        #label nodes by strip
        for sk in non_merged:
            sk.nodes['label']=sk.strip
        for sk in merged:
            sk.nodes['label']='merge'
        #replace strip files
        for ind,strip in enumerate(list(set(non_merged.strip))):
            strip_sk = [i for i, value in enumerate(non_merged.strip) if value==strip]
            subset = non_merged[strip_sk]
            subset = navis.NeuronList([subset,merged])

            os.remove(strip)
            write_navis_skels_tar(strip, subset)
    
    return non_merged,merged

<b> 2 Strips!<b>

In [ ]:
#Load model files
sc = load("/ACdata/Users/connorl/Models/scaler.joblib") #scalar file
cl = load("/ACdata/Users/connorl/Models/LR_1.joblib") #model file

In [ ]:
#Import skeletons (stage position translation already applied)
sk1 = read_navis_neurons_tar("/ACdata/Users/connorl/Skeletons/H17_x55_S33_230413_highres_MIP1/Reconnected/highres_Pos1.swcs.tar.gz")
sk2 = read_navis_neurons_tar("/ACdata/Users/connorl/Skeletons/H17_x55_S33_230413_highres_MIP1/Reconnected/highres_Pos2.swcs.tar.gz")

In [ ]:
#Apply translation/s

#Stage position
sk1 = translate_nodes(sk1, trans=[0.0, -246.3, 0.0])
sk2 = translate_nodes(sk2, trans=[0.0, -492.6, 0.0])

#Stitching
sk1 = translate_nodes(sk1, trans=[-4.208941859745392, 0.7954165880649953, 1.9973094002111451])

In [ ]:
#Run reconnection (overlap parameter indicates dimensions with overlap [x,y,z])
%time non_merged,merged = reconnect_strips(strip1=sk1, strip2=sk2, overlap=[0,1,0], cl=cl, sc=sc, edge_prop=.8, prob_thresh=None, min_nodes=10, query_dis=20, min_collin=0.7)

In [ ]:
#Convert strip names to colors
labels = list(non_merged.strip) + list(merged.strip)
comb = navis.NeuronList([non_merged,merged])
comb.set_neuron_attributes(labels, 'strip')

#Filter skeletons using arbitary cutout
filt = filter_skeletons(comb,[13500,14000,-300,-150,100,300]) #x1,x2,y1,y2,z1,z2
for sk in filt:
    sk.name = str(sk.id)

new_labels = list((pd.Series(filt.strip)).map({'strip1':'green','strip2':'magenta','merged':'brown'}))
filt.set_neuron_attributes(new_labels, 'strip')

In [ ]:
#Skeleton metrics
mean = np.mean(merged.cable_length)
median = np.median(merged.cable_length)
print(str(len(merged)) + " Merged Skeletons, ","Mean/Median Cable Length: " + str(mean) + ", " + str(median))

In [ ]:
#Plot
filt.plot3d(color = filt.strip, radius=True)

<b> 3 Strips!<b>

In [ ]:
#Import skeletons (stage position translation already applied)
sk1 = read_navis_neurons_tar("/ACdata/Users/connorl/Skeletons/H17_x55_S33_230413_highres_MIP1/Reconnected/highres_Pos1.swcs.tar.gz")
sk2 = read_navis_neurons_tar("/ACdata/Users/connorl/Skeletons/H17_x55_S33_230413_highres_MIP1/Reconnected/highres_Pos2.swcs.tar.gz")
sk3 = read_navis_neurons_tar("/ACdata/Users/connorl/Skeletons/H17_x55_S33_230413_highres_MIP1/Reconnected/highres_Pos3.swcs.tar.gz")

In [ ]:
#Apply translation/s
#Stage position
sk1 = translate_nodes(sk1, trans=[0.0, -246.3, 0.0])
sk2 = translate_nodes(sk2, trans=[0.0, -492.6, 0.0])
sk3 = translate_nodes(sk3, trans=[0.0, -738.9, 0.0])

#Stitching
#sk1 = translate_nodes(sk1, trans=[-4.208941859745392, 0.7954165880649953, 1.9973094002111451])
#sk1 = translate_nodes(sk1, trans=[-4.009026087498205, -0.602828339784878, 1.994332626764975])
#sk2 = translate_nodes(sk2, trans=[-4.009026087498205, -0.602828339784878, 1.994332626764975])

In [ ]:
sk1.set_neuron_attributes(['strip1']*int(len(sk1)), 'strip')
sk2.set_neuron_attributes(['strip2']*int(len(sk2)), 'strip')
sk3.set_neuron_attributes(['strip3']*int(len(sk3)), 'strip')
skels = navis.NeuronList([sk1,sk2,sk3])

In [ ]:
#Filter skeletons using arbitary cutout
skels = filter_skeletons(skels,[13000,14000,-650,-150,100,300])

In [ ]:
%time non_merged, merged = reconnect(skels=skels, cl=cl, sc=sc, min_nodes=10, query_dis=10, min_collin=0.7, prob_thresh=None, resample=None, smooth=None, split=False)

In [ ]:
#Convert strip names to colors
merged.set_neuron_attributes(['merged']*int(len(merged)), 'strip')
labels = list(non_merged.strip) + list(merged.strip)
comb = navis.NeuronList([non_merged,merged])
comb.set_neuron_attributes(labels, 'strip')

for sk in comb:
    sk.name = str(sk.id)

new_labels = list((pd.Series(comb.strip)).map({'strip1':'green','strip2':'magenta','strip3':'blue','merged':'brown'}))
comb.set_neuron_attributes(new_labels, 'strip')

In [ ]:
#Skeleton metrics
mean = np.mean(merged.cable_length)
median = np.median(merged.cable_length)
print(str(len(merged)) + " Merged Skeletons, ","Mean/Median Cable Length: " + str(mean) + ", " + str(median))

In [ ]:
#Plot
comb.plot3d(color = comb.strip, radius=True)